In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

['Classroom', 'hackericon.gsite', 'DriveHub', 'Sharer.pw', 'Synopsis of project under K.B. Sir.pdf', '16764817532917073073389220742982.jpg', 'PhotoRoom-20240119_011828.png', 'Presidency University(25).PDF', 'Signature  (1).jpg', 'Content (1).csv', 'IMG-20240620-WA0003.jpg', 'Screenshot_20240620_140709_Chrome.jpg', 'Document (3) (5).docx', 'download_copy.pdf', 'Colab Notebooks', 'L01--Introduction (1).gdoc', 'L01--Introduction.gdoc', 'LA Marksheet .pdf', 'LA_PRES.pdf', 'L02--Search.gdoc', '01a - linear-regression-MSE plot.py', '01b-regularisations.py', 'search.py.gdoc', 'kde-assignment.zip', 'grad.mark.pdf', '6th_sem.pdf', 'Sovan_Pal_SOP.pdf', 'Resume_sovan (2).pdf', 'ML_project.zip', 'ML_Project_Report.pdf', 'IMG-20251222-WA0001.jpg', '20250913150209-B2530088-S-removebg-preview.png', 'WhatsApp Image 2026-01-07 at 00.02.14.jpeg', 'WhatsApp Image 2026-01-07 at 00.02.14 (1).jpeg', 'WhatsApp Image 2026-01-07 at 00.02.10.pdf', 'requirements.gdoc', 'Schedule.gsheet', 'accenture_forage_certif

In [ ]:
import os

train_rgb_path = "/content/drive/MyDrive/DL Project/Kaggle_Prepared/train/RGB"

rgb_files = os.listdir(train_rgb_path)
print("Total RGB files:", len(rgb_files))
print(rgb_files[:10])

Total RGB files: 600
['Other_hyper_38.png', 'Other_hyper_128.png', 'Other_hyper_143.png', 'Rust_hyper_65.png', 'Health_hyper_184.png', 'Rust_hyper_53.png', 'Rust_hyper_139.png', 'Health_hyper_107.png', 'Other_hyper_111.png', 'Other_hyper_135.png']


## Data Preparation & Splitting

In [ ]:
health_rgb = []

for f in rgb_files:
    if "Health" in f:
        health_rgb.append(f)


rust_rgb = []

for f in rgb_files:
    if "Rust" in f:
        rust_rgb.append(f)


other_rgb = []

for f in rgb_files:
    if "Other" in f:
        other_rgb.append(f)

print(len(health_rgb), len(rust_rgb), len(other_rgb))


200 200 200


In [ ]:
print(health_rgb[:10])
print(rust_rgb[:10])
print(other_rgb[:10])

['Health_hyper_184.png', 'Health_hyper_107.png', 'Health_hyper_181.png', 'Health_hyper_96.png', 'Health_hyper_21.png', 'Health_hyper_71.png', 'Health_hyper_195.png', 'Health_hyper_171.png', 'Health_hyper_75.png', 'Health_hyper_151.png']
['Rust_hyper_65.png', 'Rust_hyper_53.png', 'Rust_hyper_139.png', 'Rust_hyper_62.png', 'Rust_hyper_161.png', 'Rust_hyper_145.png', 'Rust_hyper_10.png', 'Rust_hyper_129.png', 'Rust_hyper_41.png', 'Rust_hyper_134.png']
['Other_hyper_38.png', 'Other_hyper_128.png', 'Other_hyper_143.png', 'Other_hyper_111.png', 'Other_hyper_135.png', 'Other_hyper_65.png', 'Other_hyper_82.png', 'Other_hyper_56.png', 'Other_hyper_28.png', 'Other_hyper_52.png']


In [ ]:
import random

random.shuffle(health_rgb)
random.shuffle(rust_rgb)
random.shuffle(other_rgb)


In [ ]:
# Health Split
train_health = health_rgb[:160]
val_health   = health_rgb[160:]

# Rust Split
train_rust = rust_rgb[:160]
val_rust   = rust_rgb[160:]

# Other Split
train_other = other_rgb[:160]
val_other   = other_rgb[160:]


print(len(train_health), len(val_health))
print(len(train_rust), len(val_rust))
print(len(train_other), len(val_other))


160 40
160 40
160 40


In [ ]:
train_files = train_health + train_rust + train_other
val_files   = val_health + val_rust + val_other

In [ ]:
train_data = []

# Health = 0
for f in train_health:
    train_data.append((f, 0))

# Rust = 1
for f in train_rust:
    train_data.append((f, 1))

# Other = 2
for f in train_other:
    train_data.append((f, 2))

print(len(train_data))


480


In [ ]:
val_data = []

# Health = 0
for f in val_health:
    val_data.append((f, 0))

# Rust = 1
for f in val_rust:
    val_data.append((f, 1))

# Other = 2
for f in val_other:
    val_data.append((f, 2))

print(len(val_data))


120


In [ ]:
# But inside train_data, the order is:

# All Health (160)
# All Rust (160)
# All Other (160)
# That is NOT ideal for training.


random.shuffle(train_data)
random.shuffle(val_data)

CNN MODEL for RGB Images

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import os
from sklearn.metrics import accuracy_score , classification_report

In [ ]:
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
class RGBDataset(Dataset):
    def __init__(self, folder, file_list, transform=None):
        self.folder = folder
        self.files = sorted(file_list )
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        path = os.path.join(self.folder, file)

        image = Image.open(path).convert("RGB")

        file_lower = file.lower()

        if "health" in file_lower:
            label = 0
        elif "rust" in file_lower:
            label = 1
        elif "other" in file_lower:
            label = 2
        else:
            raise ValueError(f"Unknown label: {file}")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),                         # crop instead of direct resize
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),                 # rare grayscale helps generalise
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2),                    # randomly masks patches
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
train_dataset = RGBDataset(train_rgb_path, train_files, train_transform)
val_dataset   = RGBDataset(train_rgb_path, val_files,   val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_dataset)}  |  Val samples: {len(val_dataset)}")


Train samples: 480  |  Val samples: 120


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Unfreeze ALL layers from the start — dataset is large enough
# and we use a very small LR for backbone to avoid destroying weights
for param in model.parameters():
    param.requires_grad = True

# Replace classifier head
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.BatchNorm1d(256),          # stabilises training
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 3)
)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable_params:,} / {total_params:,}")

Trainable params: 11,309,123 / 11,309,123


In [ ]:
def train_model(model, train_loader, val_loader, epochs=25):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # smoother targets → less overconfidence

    # Differential LR: backbone gets 10× lower LR than head
    backbone_params = [p for n, p in model.named_parameters() if "fc" not in n]
    head_params     = list(model.fc.parameters())

    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": 1e-4},   # fine-tune backbone gently
        {"params": head_params,     "lr": 1e-3},   # train head faster
    ], weight_decay=1e-4)

    # Cosine annealing: smoothly decays LR → prevents oscillation
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_acc   = 0.0
    best_epoch = 0

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()

        # ── Validate ──
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                preds = torch.argmax(model(images), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc      = accuracy_score(all_labels, all_preds)
        avg_loss = train_loss / len(train_loader)
        current_lr = scheduler.get_last_lr()[0]

        if acc > best_acc:
            best_acc   = acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), "best_model.pth")
            marker = "  ← best"
        else:
            marker = ""

        print(f"Epoch {epoch+1:>2}/{epochs}  |  "
              f"Loss: {avg_loss:.4f}  |  "
              f"Val Acc: {acc:.4f}  |  "
              f"LR: {current_lr:.2e}"
              + marker)

    print(f"\nBest Val Accuracy: {best_acc:.4f}  (Epoch {best_epoch})")


In [ ]:
def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds   = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n── Final Evaluation ──")
    print(classification_report(
        all_labels, all_preds,
        target_names=["Health", "Rust", "Other"]
    ))

In [ ]:
train_model(model, train_loader, val_loader, epochs=15)

# Load best checkpoint and evaluate
model.load_state_dict(torch.load("best_model.pth"))
evaluate(model, val_loader)


Epoch  1/15  |  Loss: 1.0808  |  Val Acc: 0.5667  |  LR: 9.89e-05  ← best
Epoch  2/15  |  Loss: 0.9990  |  Val Acc: 0.5250  |  LR: 9.57e-05
Epoch  3/15  |  Loss: 0.9867  |  Val Acc: 0.5417  |  LR: 9.05e-05
Epoch  4/15  |  Loss: 0.9183  |  Val Acc: 0.5667  |  LR: 8.36e-05
Epoch  5/15  |  Loss: 0.9308  |  Val Acc: 0.5833  |  LR: 7.52e-05  ← best
Epoch  6/15  |  Loss: 0.9092  |  Val Acc: 0.5583  |  LR: 6.58e-05
Epoch  7/15  |  Loss: 0.8705  |  Val Acc: 0.5750  |  LR: 5.57e-05
Epoch  8/15  |  Loss: 0.8561  |  Val Acc: 0.5833  |  LR: 4.53e-05
Epoch  9/15  |  Loss: 0.8488  |  Val Acc: 0.5917  |  LR: 3.52e-05  ← best
Epoch 10/15  |  Loss: 0.8109  |  Val Acc: 0.6167  |  LR: 2.58e-05  ← best
Epoch 11/15  |  Loss: 0.7835  |  Val Acc: 0.5917  |  LR: 1.74e-05
Epoch 12/15  |  Loss: 0.7788  |  Val Acc: 0.6250  |  LR: 1.05e-05  ← best
Epoch 13/15  |  Loss: 0.7773  |  Val Acc: 0.6000  |  LR: 5.28e-06
Epoch 14/15  |  Loss: 0.7929  |  Val Acc: 0.6083  |  LR: 2.08e-06
Epoch 15/15  |  Loss: 0.7987  |  Val

CNN MODEL for MS Data

In [ ]:
import os
ms_path = "/content/drive/MyDrive/DL Project/Kaggle_Prepared/train/MS"
files = os.listdir(ms_path)
print(f"Total files: {len(files)}")
print("First 10 files:", files[:10])
print("Extension:", os.path.splitext(files[0])[1])

Total files: 600
First 10 files: ['Rust_hyper_46.tif', 'Rust_hyper_22.tif', 'Rust_hyper_41.tif', 'Rust_hyper_181.tif', 'Rust_hyper_42.tif', 'Rust_hyper_195.tif', 'Other_hyper_67.tif', 'Rust_hyper_108.tif', 'Rust_hyper_5.tif', 'Other_hyper_65.tif']
Extension: .tif


In [ ]:
import numpy as np
sample = os.path.join(ms_path, files[0])

ext = os.path.splitext(files[0])[1].lower()
if ext == ".npy":
    arr = np.load(sample)
    print("Shape:", arr.shape)
    print("Dtype:", arr.dtype)
    print("Min/Max:", arr.min(), arr.max())
elif ext in (".tif", ".tiff"):
    import rasterio
    with rasterio.open(sample) as src:
        print("Bands:", src.count)
        print("Shape:", src.height, src.width)
        print("Dtype:", src.dtypes)
elif ext == ".png":
    from PIL import Image
    img = Image.open(sample)
    print("Mode:", img.mode)
    print("Size:", img.size)

Bands: 5
Shape: 64 64
Dtype: ('uint16', 'uint16', 'uint16', 'uint16', 'uint16')


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


In [ ]:
import rasterio

In [ ]:
# Path
ms_path = "/content/drive/MyDrive/DL Project/Kaggle_Prepared/train/MS"

In [ ]:
# ── File split ────────────────────────────────────────────────
ms_files = os.listdir(ms_path)
health_ms, rust_ms, other_ms = [], [], []
for f in ms_files:
    if   "Health" in f: health_ms.append(f)
    elif "Rust"   in f: rust_ms.append(f)
    elif "Other"  in f: other_ms.append(f)

print(f"Health: {len(health_ms)}  Rust: {len(rust_ms)}  Other: {len(other_ms)}")

random.seed(42)
random.shuffle(health_ms)
random.shuffle(rust_ms)
random.shuffle(other_ms)

train_files = health_ms[:160] + rust_ms[:160] + other_ms[:160]
val_files   = health_ms[160:] + rust_ms[160:] + other_ms[160:]
random.shuffle(train_files)
random.shuffle(val_files)
print(f"Train: {len(train_files)}  |  Val: {len(val_files)}")

Health: 200  Rust: 200  Other: 200
Train: 480  |  Val: 120


In [ ]:
class MSDataset(Dataset):
    def __init__(self, folder, file_list, mean, std, augment=False):
        self.folder  = folder
        self.files   = sorted(file_list)
        self.mean    = torch.tensor(mean, dtype=torch.float32).view(5, 1, 1)
        self.std     = torch.tensor(std,  dtype=torch.float32).view(5, 1, 1)
        self.augment = augment
        self.resize  = transforms.Resize(
            (112, 112),                        # 112 instead of 224 — less overfitting
            interpolation=transforms.InterpolationMode.BILINEAR,
            antialias=True
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        path = os.path.join(self.folder, file)

        file_lower = file.lower()
        if   "health" in file_lower: label = 0
        elif "rust"   in file_lower: label = 1
        elif "other"  in file_lower: label = 2
        else: raise ValueError(f"Unknown label: {file}")

        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32) / 65535.0   # (5, 64, 64)

        tensor = torch.from_numpy(arr)
        tensor = self.resize(tensor)                         # (5, 112, 112)
        tensor = (tensor - self.mean) / (self.std + 1e-6)   # normalise

        if self.augment:
            # Spatial augmentations
            if random.random() > 0.5:
                tensor = torch.flip(tensor, dims=[2])        # h-flip
            if random.random() > 0.5:
                tensor = torch.flip(tensor, dims=[1])        # v-flip
            k = random.randint(0, 3)
            if k > 0:
                tensor = torch.rot90(tensor, k, dims=[1, 2])

            # Spectral augmentation — randomly scale individual bands slightly
            # Simulates sensor noise and varying illumination per band
            scale = torch.empty(5, 1, 1).uniform_(0.9, 1.1)
            tensor = tensor * scale

            # Random band dropout — force model to not rely on any single band
            # Randomly zero out one band with 20% probability
            if random.random() < 0.2:
                band_idx = random.randint(0, 4)
                tensor[band_idx] = 0.0

        return tensor, label

In [ ]:
print("\nComputing per-band statistics...")
channel_sum  = np.zeros(5, dtype=np.float64)
channel_sum2 = np.zeros(5, dtype=np.float64)
n_pixels     = 0

for f in train_files:
    with rasterio.open(os.path.join(ms_path, f)) as src:
        arr = src.read().astype(np.float64) / 65535.0
    channel_sum  += arr.sum(axis=(1, 2))
    channel_sum2 += (arr ** 2).sum(axis=(1, 2))
    n_pixels     += arr.shape[1] * arr.shape[2]

MEAN = (channel_sum  / n_pixels).tolist()
STD  = (np.sqrt(channel_sum2 / n_pixels - (channel_sum / n_pixels) ** 2)).tolist()
print(f"Mean: {[round(m,4) for m in MEAN]}")
print(f"Std : {[round(s,4) for s in STD]}")


Computing per-band statistics...
Mean: [0.0064, 0.0117, 0.0123, 0.0334, 0.0411]
Std : [0.006, 0.0075, 0.0105, 0.0133, 0.0174]


In [ ]:
# ── Spectral Attention Module ─────────────────────────────────
class SpectralAttention(nn.Module):
    """
    Learns which of the 5 spectral bands are most informative
    for each sample. Acts like a soft band selector.

    For plant disease:
      - NIR (band 4) → chlorophyll content
      - RedEdge (band 3) → early stress detection
      - Red (band 2) → pigmentation changes
    The model learns these weights automatically.
    """
    def __init__(self, in_channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),              # global average per band
            nn.Flatten(),
            nn.Linear(in_channels, in_channels),
            nn.ReLU(),
            nn.Linear(in_channels, in_channels),
            nn.Sigmoid()                          # weights between 0–1 per band
        )

    def forward(self, x):
        # x: (B, C, H, W)
        weights = self.attention(x)               # (B, C)
        weights = weights.view(weights.shape[0], weights.shape[1], 1, 1)
        return x * weights                        # reweight each band



In [ ]:
# ── Spectral Feature Extractor ────────────────────────────────
class SpectralEncoder(nn.Module):
    """
    Processes each spectral band independently first,
    then fuses them — unlike ResNet which mixes bands immediately
    in the first conv layer.

    This respects the physical meaning of each spectral band.
    """
    def __init__(self, in_channels=5):
        super().__init__()

        # Per-band feature extraction (shared weights across bands)
        self.band_encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 1),        # 1×1 conv = per-pixel spectral mixing
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        # Spectral attention — learn which bands matter
        self.spectral_attn = SpectralAttention(in_channels)

        # Attended input goes into this projection
        self.input_proj = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        # Fuse band_encoder + attended projection → 64 channels
        self.fuse = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 3, 1),                  # project to 3ch for ResNet backbone
            nn.BatchNorm2d(3),
            nn.ReLU(),
        )

    def forward(self, x):
        # x: (B, 5, H, W)
        attended = self.spectral_attn(x)           # (B, 5, H, W) — reweighted bands
        proj     = self.input_proj(attended)        # (B, 32, H, W)
        encoded  = self.band_encoder(x)             # (B, 32, H, W)
        fused    = self.fuse(torch.cat([proj, encoded], dim=1))  # (B, 3, H, W)
        return fused                                # (B, 3, H, W) → into ResNet



In [ ]:
# ── Full Model ────────────────────────────────────────────────
class MSModel(nn.Module):
    def __init__(self, in_channels=5, num_classes=3):
        super().__init__()

        # Spectral front-end — learns to extract meaningful spectral features
        self.spectral_encoder = SpectralEncoder(in_channels)

        # ResNet18 backbone — takes the 3ch output from spectral encoder
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Remove final FC layer, keep feature extractor
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # (B, 512, 1, 1)

        # Classifier head with strong regularisation
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.spectral_encoder(x)    # (B, 5, H, W) → (B, 3, H, W)
        x = self.backbone(x)            # (B, 3, H, W) → (B, 512, 1, 1)
        x = self.classifier(x)          # (B, 512) → (B, 3)
        return x


In [ ]:
# ── DataLoaders ───────────────────────────────────────────────
train_dataset = MSDataset(ms_path, train_files, MEAN, STD, augment=True)
val_dataset   = MSDataset(ms_path, val_files,   MEAN, STD, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

imgs, lbls = next(iter(train_loader))
print(f"\nBatch shape : {imgs.shape}")    # (32, 5, 112, 112)
print(f"Labels      : {lbls.tolist()}")


Batch shape : torch.Size([32, 5, 112, 112])
Labels      : [0, 1, 1, 2, 0, 1, 1, 0, 1, 2, 1, 1, 0, 0, 1, 0, 2, 2, 0, 2, 2, 2, 2, 2, 0, 1, 2, 1, 1, 1, 0, 1]


In [ ]:
# ── Build model ───────────────────────────────────────────────
model = MSModel(in_channels=5, num_classes=3).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 139MB/s]


Total parameters: 11,382,120


In [ ]:
# ── Training ──────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs=40):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Three param groups with different LRs
    spectral_params  = list(model.spectral_encoder.parameters())
    backbone_params  = list(model.backbone.parameters())
    head_params      = list(model.classifier.parameters())

    optimizer = optim.AdamW([
        {"params": spectral_params, "lr": 5e-4},   # train spectral encoder faster
        {"params": backbone_params, "lr": 5e-5},   # fine-tune backbone gently
        {"params": head_params,     "lr": 1e-3},   # train head fastest
    ], weight_decay=2e-4)                           # stronger L2 regularisation

    # Warm restarts — helps escape local minima, better for small datasets
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=1, eta_min=1e-6
    )

    best_acc   = 0.0
    best_epoch = 0
    patience   = 15
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                preds = torch.argmax(model(images), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc      = accuracy_score(all_labels, all_preds)
        avg_loss = train_loss / len(train_loader)
        cur_lr   = optimizer.param_groups[2]["lr"]   # head LR

        if acc > best_acc:
            best_acc   = acc
            best_epoch = epoch + 1
            no_improve = 0
            torch.save(model.state_dict(), "best_ms_v2.pth")
            marker = "  ← best"
        else:
            no_improve += 1
            marker = ""

        print(f"Epoch {epoch+1:>2}/{epochs}  |  "
              f"Loss: {avg_loss:.4f}  |  "
              f"Val Acc: {acc:.4f}  |  "
              f"LR: {cur_lr:.2e}" + marker)

        if no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    print(f"\nBest Val Accuracy: {best_acc:.4f}  (Epoch {best_epoch})")


def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            preds = torch.argmax(model(images), dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n── Multispectral v2 Final Evaluation ──")
    print(classification_report(
        all_labels, all_preds,
        target_names=["Health", "Rust", "Other"]
    ))


In [ ]:
import warnings
import rasterio
import rasterio.errors
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

# Also suppress in worker processes
import os
os.environ["RASTERIO_IGNORE_GEOREF_WARNING"] = "1"

In [ ]:
# ── Run ───────────────────────────────────────────────────────
train_model(model, train_loader, val_loader, epochs=40)
model.load_state_dict(torch.load("best_ms_v2.pth"))
evaluate(model, val_loader)

Epoch  1/40  |  Loss: 0.8820  |  Val Acc: 0.5750  |  LR: 9.76e-04  ← best
Epoch  2/40  |  Loss: 0.8787  |  Val Acc: 0.5667  |  LR: 9.05e-04
Epoch  3/40  |  Loss: 0.8542  |  Val Acc: 0.5667  |  LR: 7.94e-04
Epoch  4/40  |  Loss: 0.8237  |  Val Acc: 0.5833  |  LR: 6.55e-04  ← best
Epoch  5/40  |  Loss: 0.8416  |  Val Acc: 0.5167  |  LR: 5.01e-04
Epoch  6/40  |  Loss: 0.8290  |  Val Acc: 0.5750  |  LR: 3.46e-04
Epoch  7/40  |  Loss: 0.7994  |  Val Acc: 0.5667  |  LR: 2.07e-04
Epoch  8/40  |  Loss: 0.7965  |  Val Acc: 0.5833  |  LR: 9.64e-05
Epoch  9/40  |  Loss: 0.7867  |  Val Acc: 0.5583  |  LR: 2.54e-05
Epoch 10/40  |  Loss: 0.7964  |  Val Acc: 0.5833  |  LR: 1.00e-03
Epoch 11/40  |  Loss: 0.8160  |  Val Acc: 0.5583  |  LR: 9.76e-04
Epoch 12/40  |  Loss: 0.7733  |  Val Acc: 0.5583  |  LR: 9.05e-04
Epoch 13/40  |  Loss: 0.7752  |  Val Acc: 0.5417  |  LR: 7.94e-04
Epoch 14/40  |  Loss: 0.7870  |  Val Acc: 0.5667  |  LR: 6.55e-04
Epoch 15/40  |  Loss: 0.7813  |  Val Acc: 0.5500  |  LR: 5.0

In [ ]:
# ── Model: ResNet18 with 5-channel input ──────────────────────
def build_ms_resnet(in_channels=5, num_classes=3):
    model    = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    old_conv = model.conv1

    new_conv = nn.Conv2d(
        in_channels,
        old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )

    with torch.no_grad():
        # Copy pretrained RGB weights into first 3 channels
        new_conv.weight[:, :3, :, :] = old_conv.weight
        # Initialise extra channels (4, 5) as mean of RGB weights
        extra = old_conv.weight.mean(dim=1, keepdim=True)
        for i in range(3, in_channels):
            new_conv.weight[:, i:i+1, :, :] = extra

    model.conv1 = new_conv

    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model

model = build_ms_resnet(in_channels=5, num_classes=3).to(device)
print(f"\nModel ready — input channels: 5")


Model ready — input channels: 5


In [ ]:
# ── Training ──────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs=30):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    backbone_params = [p for n, p in model.named_parameters() if "fc" not in n]
    head_params     = list(model.fc.parameters())

    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": 1e-4},
        {"params": head_params,     "lr": 1e-3},
    ], weight_decay=1e-4)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    best_acc   = 0.0
    best_epoch = 0
    patience   = 10
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                preds = torch.argmax(model(images), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc      = accuracy_score(all_labels, all_preds)
        avg_loss = train_loss / len(train_loader)
        cur_lr   = optimizer.param_groups[1]["lr"]

        if acc > best_acc:
            best_acc   = acc
            best_epoch = epoch + 1
            no_improve = 0
            torch.save(model.state_dict(), "best_ms_model.pth")
            marker = "  ← best"
        else:
            no_improve += 1
            marker = ""

        print(f"Epoch {epoch+1:>2}/{epochs}  |  "
              f"Loss: {avg_loss:.4f}  |  "
              f"Val Acc: {acc:.4f}  |  "
              f"LR: {cur_lr:.2e}" + marker)

        if no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    print(f"\nBest Val Accuracy: {best_acc:.4f}  (Epoch {best_epoch})")


def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            preds = torch.argmax(model(images), dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\n── Multispectral Final Evaluation ──")
    print(classification_report(
        all_labels, all_preds,
        target_names=["Health", "Rust", "Other"]
    ))


In [ ]:
import warnings
import rasterio.errors
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

In [ ]:
# ── Run ───────────────────────────────────────────────────────
train_model(model, train_loader, val_loader, epochs=30)
model.load_state_dict(torch.load("best_ms_model.pth"))
evaluate(model, val_loader)

Epoch  1/30  |  Loss: 0.7659  |  Val Acc: 0.4333  |  LR: 9.97e-04  ← best
Epoch  2/30  |  Loss: 0.7102  |  Val Acc: 0.5167  |  LR: 9.89e-04  ← best
Epoch  3/30  |  Loss: 0.6595  |  Val Acc: 0.5250  |  LR: 9.76e-04  ← best
Epoch  4/30  |  Loss: 0.6749  |  Val Acc: 0.5583  |  LR: 9.57e-04  ← best
Epoch  5/30  |  Loss: 0.6976  |  Val Acc: 0.5667  |  LR: 9.33e-04  ← best
Epoch  6/30  |  Loss: 0.6446  |  Val Acc: 0.5083  |  LR: 9.05e-04
Epoch  7/30  |  Loss: 0.6308  |  Val Acc: 0.5750  |  LR: 8.72e-04  ← best
Epoch  8/30  |  Loss: 0.6252  |  Val Acc: 0.5000  |  LR: 8.35e-04
Epoch  9/30  |  Loss: 0.5990  |  Val Acc: 0.5250  |  LR: 7.94e-04
Epoch 10/30  |  Loss: 0.5885  |  Val Acc: 0.5583  |  LR: 7.50e-04
Epoch 11/30  |  Loss: 0.5372  |  Val Acc: 0.5667  |  LR: 7.04e-04
Epoch 12/30  |  Loss: 0.4980  |  Val Acc: 0.5500  |  LR: 6.55e-04
Epoch 13/30  |  Loss: 0.5301  |  Val Acc: 0.5583  |  LR: 6.04e-04
Epoch 14/30  |  Loss: 0.5330  |  Val Acc: 0.5917  |  LR: 5.53e-04  ← best
Epoch 15/30  |  Loss

CNN for HS Data

In [ ]:
import warnings
import rasterio
from rasterio.errors import NotGeoreferencedWarning
import numpy as np
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

In [ ]:
import os, rasterio

# Find the hyperspectral folder
base = "/content/drive/MyDrive/DL Project/Kaggle_Prepared/train"
print("All folders:", os.listdir(base))

All folders: ['.DS_Store', 'MS', 'RGB', 'HS']


In [ ]:
hs_path = "/content/drive/MyDrive/DL Project/Kaggle_Prepared/train/HS"
# update name if different

files = os.listdir(hs_path)
print(f"Total files: {len(files)}")
print("Sample files:", files[:5])

# Check one file
with rasterio.open(os.path.join(hs_path, files[0])) as src:
    print(f"\nBands  : {src.count}")
    print(f"Shape  : {src.height} x {src.width}")
    print(f"Dtype  : {src.dtypes[0]}")

Total files: 600
Sample files: ['Rust_hyper_84.tif', 'Rust_hyper_63.tif', 'Rust_hyper_80.tif', 'Rust_hyper_73.tif', 'Rust_hyper_50.tif']

Bands  : 125
Shape  : 32 x 32
Dtype  : uint16


In [ ]:
# ── File split ────────────────────────────────────────────────
hs_files = os.listdir(hs_path)
health_hs, rust_hs, other_hs = [], [], []
for f in hs_files:
    if   "Health" in f: health_hs.append(f)
    elif "Rust"   in f: rust_hs.append(f)
    elif "Other"  in f: other_hs.append(f)

print(f"Health: {len(health_hs)}  Rust: {len(rust_hs)}  Other: {len(other_hs)}")

random.seed(42)
random.shuffle(health_hs)
random.shuffle(rust_hs)
random.shuffle(other_hs)

train_files = health_hs[:160] + rust_hs[:160] + other_hs[:160]
val_files   = health_hs[160:] + rust_hs[160:] + other_hs[160:]
random.shuffle(train_files)
random.shuffle(val_files)
print(f"Train: {len(train_files)}  |  Val: {len(val_files)}")




Health: 200  Rust: 200  Other: 200
Train: 480  |  Val: 120


In [ ]:
# ── Check band counts across all files ───────────────────────
print("Checking band counts across all files...")
band_counts = {}
for f in hs_files:
    with rasterio.open(os.path.join(hs_path, f)) as src:
        n = src.count
    band_counts[n] = band_counts.get(n, 0) + 1

print("Band count distribution:", band_counts)
# e.g. {125: 598, 126: 2} — tells you how many files have each count

# Use the most common band count as N_BANDS
N_BANDS = max(band_counts, key=band_counts.get)
print(f"Using N_BANDS = {N_BANDS} (most common)")

# ── Compute dataset statistics (robust to inconsistent bands) ─
print(f"\nComputing per-band statistics ({N_BANDS} bands)...")
channel_sum  = np.zeros(N_BANDS, dtype=np.float64)
channel_sum2 = np.zeros(N_BANDS, dtype=np.float64)
n_pixels     = 0
skipped      = 0

for f in train_files:
    with rasterio.open(os.path.join(hs_path, f)) as src:
        arr = src.read().astype(np.float64) / 65535.0   # (B, 32, 32)

    # Truncate to N_BANDS if file has more
    if arr.shape[0] > N_BANDS:
        arr = arr[:N_BANDS]
    # Skip files with fewer bands than expected
    elif arr.shape[0] < N_BANDS:
        skipped += 1
        continue

    channel_sum  += arr.sum(axis=(1, 2))
    channel_sum2 += (arr ** 2).sum(axis=(1, 2))
    n_pixels     += arr.shape[1] * arr.shape[2]

if skipped > 0:
    print(f"Skipped {skipped} files with fewer than {N_BANDS} bands")

MEAN = channel_sum  / n_pixels
STD  = np.sqrt(channel_sum2 / n_pixels - MEAN ** 2)
print(f"Mean range: [{MEAN.min():.4f}, {MEAN.max():.4f}]")
print(f"Std  range: [{STD.min():.4f},  {STD.max():.4f}]")
print("Statistics computed successfully!")


Checking band counts across all files...
Band count distribution: {125: 489, 126: 111}
Using N_BANDS = 125 (most common)

Computing per-band statistics (125 bands)...
Mean range: [0.0041, 0.0415]
Std  range: [0.0050,  0.0170]
Statistics computed successfully!


In [ ]:
# ── Dataset v3 — stronger spectral augmentation ─────────────
class HSDataset(Dataset):
    def __init__(self, folder, file_list, mean, std, augment=False):
        self.folder  = folder
        self.files   = sorted(file_list)
        self.mean    = torch.tensor(mean, dtype=torch.float32).view(N_BANDS, 1, 1)
        self.std     = torch.tensor(std,  dtype=torch.float32).view(N_BANDS, 1, 1)
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        path = os.path.join(self.folder, file)

        file_lower = file.lower()
        if   'health' in file_lower: label = 0
        elif 'rust'   in file_lower: label = 1
        elif 'other'  in file_lower: label = 2
        else: raise ValueError(f'Unknown label: {file}')

        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32) / 65535.0

        if arr.shape[0] > N_BANDS:
            arr = arr[:N_BANDS]

        tensor = torch.from_numpy(arr)
        tensor = (tensor - self.mean) / (self.std + 1e-6)

        if self.augment:
            # Spatial flips & rotation
            if random.random() > 0.5:
                tensor = torch.flip(tensor, dims=[2])
            if random.random() > 0.5:
                tensor = torch.flip(tensor, dims=[1])
            k = random.randint(0, 3)
            if k > 0:
                tensor = torch.rot90(tensor, k, dims=[1, 2])

            # Gaussian noise
            noise = torch.randn_like(tensor) * 0.03
            tensor = tensor + noise

            # Spectral band scaling (simulates sensor variation)
            scale = torch.empty(N_BANDS, 1, 1).uniform_(0.85, 1.15)
            tensor = tensor * scale

            # Band dropout — drop 0–8 random bands
            n_drop = random.randint(0, 8)
            for b in random.sample(range(N_BANDS), n_drop):
                tensor[b] = 0.0

            # Random spectral shift — circular shift along band axis
            if random.random() < 0.3:
                shift = random.randint(-5, 5)
                tensor = torch.roll(tensor, shift, dims=0)

        return tensor, label


In [ ]:
# ── Spectral Self-Attention Block ─────────────────────────────
class SpectralSelfAttention(nn.Module):
    """
    Learns inter-band relationships via a lightweight self-attention.
    Operates on the spectral dimension after global spatial pooling.
    Helps the model learn which wavelengths co-activate for each class.
    """
    def __init__(self, n_bands, reduction=8):
        super().__init__()
        mid = max(n_bands // reduction, 16)
        self.fc = nn.Sequential(
            nn.Linear(n_bands, mid),
            nn.ReLU(),
            nn.Linear(mid, n_bands),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (B, C, H, W)
        w = x.mean(dim=[2, 3])          # (B, C) global spatial pool
        w = self.fc(w).unsqueeze(-1).unsqueeze(-1)  # (B, C, 1, 1)
        return x * w


# ── Model v3 ──────────────────────────────────────────────────────
class HybridHSModel(nn.Module):
    def __init__(self, n_bands=125, num_classes=3):
        super().__init__()

        # Input spectral attention — reweights bands before any processing
        self.input_attn = SpectralSelfAttention(n_bands)

        # ── Spectral branch (1D CNN on mean spectrum) ──
        self.spectral_branch = nn.Sequential(
            nn.Conv1d(1, 32,  kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                              # 125 → 62

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),                              # 62 → 31

            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),                      # → 256*4 = 1024

            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # ── Spatial branch (2D CNN) ──
        self.spatial_branch = nn.Sequential(
            # Spectral reduction: 125 → 64 → 32
            nn.Conv2d(n_bands, 64, kernel_size=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # Spatial extraction on 32×32
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 32 → 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(2),                              # 16 → 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # ── Fusion: spectral(256) + spatial(256) = 512 ──
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.input_attn(x)                      # attend bands first
        x_spec = x.mean(dim=[2, 3]).unsqueeze(1)    # (B, 1, 125)
        x_spec = self.spectral_branch(x_spec)       # (B, 256)
        x_spat = self.spatial_branch(x)             # (B, 256)
        return self.classifier(torch.cat([x_spec, x_spat], dim=1))


In [ ]:
# ── DataLoaders ───────────────────────────────────────────────
train_dataset = HSDataset(hs_path, train_files, MEAN, STD, augment=True)
val_dataset   = HSDataset(hs_path, val_files,   MEAN, STD, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=0, pin_memory=True)

imgs, lbls = next(iter(train_loader))
print(f"Batch shape : {imgs.shape}")
print(f"Value range : [{imgs.min():.2f}, {imgs.max():.2f}]")

Batch shape : torch.Size([32, 125, 32, 32])
Value range : [-3.13, 10.81]


In [ ]:
# ── Build model ───────────────────────────────────────────────
model = HybridHSModel(n_bands=N_BANDS, num_classes=3).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")


Total parameters: 1,607,632


In [ ]:
# ── Mixup helper ──────────────────────────────────────────────
def mixup_batch(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam


# ── Training v3 ───────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs=80):
    # Class-weighted loss: Health is hardest → boost its weight
    class_weights = torch.tensor([1.8, 1.0, 1.0], device=device)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=0.05
    )

    optimizer = optim.AdamW(
        model.parameters(), lr=2e-4, weight_decay=4e-4
    )

    # Cosine annealing with warm restarts — T_0=20, T_mult=2
    # Schedule: restart at epoch 20, then 40 epochs, then 80 epochs
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=5e-7
    )

    best_acc   = 0.0
    best_epoch = 0
    patience   = 30      # higher patience to ride warm restarts
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        use_mixup  = epoch >= 5   # skip mixup in early warmup epochs

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            if use_mixup and random.random() < 0.5:
                x_mix, ya, yb, lam = mixup_batch(images, labels)
                out  = model(x_mix)
                loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)
            else:
                loss = criterion(model(images), labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                preds = torch.argmax(model(images), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc      = accuracy_score(all_labels, all_preds)
        avg_loss = train_loss / len(train_loader)
        cur_lr   = optimizer.param_groups[0]['lr']

        if acc > best_acc:
            best_acc   = acc
            best_epoch = epoch + 1
            no_improve = 0
            torch.save(model.state_dict(), 'best_hs_v3.pth')
            marker = '  ← best'
        else:
            no_improve += 1
            marker = ''

        print(f'Epoch {epoch+1:>2}/{epochs}  |  '
              f'Loss: {avg_loss:.4f}  |  '
              f'Val Acc: {acc:.4f}  |  '
              f'LR: {cur_lr:.2e}' + marker)

        if no_improve >= patience:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

    print(f'\nBest Val Accuracy: {best_acc:.4f}  (Epoch {best_epoch})')


def evaluate(model, dataloader, tta=True):
    """
    Evaluate with optional Test-Time Augmentation (TTA).
    TTA: average predictions over 4 flips — boosts accuracy ~1-2% for free.
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            if tta:
                # 4 TTA versions: original + 3 flips
                augmented = [
                    images,
                    torch.flip(images, dims=[3]),           # h-flip
                    torch.flip(images, dims=[2]),           # v-flip
                    torch.flip(images, dims=[2, 3]),        # both
                ]
                logits = sum(model(x) for x in augmented) / 4
            else:
                logits = model(images)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print('\n── Hyperspectral v3 Final Evaluation (with TTA) ──')
    print(classification_report(
        all_labels, all_preds,
        target_names=['Health', 'Rust', 'Other']
    ))


In [ ]:
# ========================================================= RUN ===========================================================
model = HybridHSModel(n_bands=N_BANDS, num_classes=3).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

train_model(model, train_loader, val_loader, epochs=80)
model.load_state_dict(torch.load('best_hs_v3.pth'))
evaluate(model, val_loader, tta=True)


Total parameters: 1,607,632
Epoch  1/80  |  Loss: 1.0722  |  Val Acc: 0.3333  |  LR: 1.99e-04  ← best
Epoch  2/80  |  Loss: 0.9832  |  Val Acc: 0.3333  |  LR: 1.95e-04
Epoch  3/80  |  Loss: 0.9354  |  Val Acc: 0.5083  |  LR: 1.89e-04  ← best
Epoch  4/80  |  Loss: 0.9004  |  Val Acc: 0.5083  |  LR: 1.81e-04
Epoch  5/80  |  Loss: 0.9063  |  Val Acc: 0.6250  |  LR: 1.71e-04  ← best
Epoch  6/80  |  Loss: 0.9274  |  Val Acc: 0.6583  |  LR: 1.59e-04  ← best
Epoch  7/80  |  Loss: 0.9032  |  Val Acc: 0.6667  |  LR: 1.46e-04  ← best
Epoch  8/80  |  Loss: 0.8706  |  Val Acc: 0.6667  |  LR: 1.31e-04
Epoch  9/80  |  Loss: 0.9030  |  Val Acc: 0.6000  |  LR: 1.16e-04
Epoch 10/80  |  Loss: 0.8665  |  Val Acc: 0.5667  |  LR: 1.00e-04
Epoch 11/80  |  Loss: 0.8757  |  Val Acc: 0.5667  |  LR: 8.46e-05
Epoch 12/80  |  Loss: 0.8861  |  Val Acc: 0.5917  |  LR: 6.94e-05
Epoch 13/80  |  Loss: 0.8503  |  Val Acc: 0.5917  |  LR: 5.50e-05
Epoch 14/80  |  Loss: 0.8569  |  Val Acc: 0.6000  |  LR: 4.16e-05
Epoch 15